In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE

df = pd.read_csv('../Set2_Employee_Attrition.csv')
df.head()


,EmployeeNumber,Age,Gender,Department,JobRole,MaritalStatus,MonthlyIncome,YearsAtCompany,OverTime,JobSatisfaction,DistanceFromHome,Attrition
0,1,58,Female,R&D,Analyst,Divorced,8090,9,Yes,1,4,No
1,2,48,Male,R&D,Rep,Married,8082,6,Yes,4,15,No
2,3,34,Female,R&D,Rep,Single,19018,4,No,4,9,No
3,4,27,Female,Sales,Manager,Single,3925,0,Yes,3,5,Yes
4,5,40,Male,Sales,Executive,Divorced,8917,12,No,4,13,No


In [13]:
df = df.drop('EmployeeNumber', axis=1)

cat_cols = df.select_dtypes(include=['object', 'str']).columns.drop('Attrition')
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

X = df.drop('Attrition', axis=1)
y = df['Attrition']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [14]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred = knn.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

Accuracy: 0.74
Precision: 0.2727272727272727
Recall: 0.14285714285714285
F1: 0.1875


In [15]:
results = []
for k in [3, 5, 7, 9, 11]:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    f1 = f1_score(y_test, pred)
    results.append((k, f1))
    print(f"K={k} -> F1: {f1:.4f}")

best_k = max(results, key=lambda x: x[1])[0]
print("Best K:", best_k)

K=3 -> F1: 0.2222
K=5 -> F1: 0.1875
K=7 -> F1: 0.1333
K=9 -> F1: 0.0800
K=11 -> F1: 0.0833
Best K: 3


In [16]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)

knn_sm = KNeighborsClassifier(n_neighbors=best_k)
knn_sm.fit(X_train_sm, y_train_sm)
y_pred_sm = knn_sm.predict(X_test_scaled)

print("After SMOTE:")
print("Accuracy:", accuracy_score(y_test, y_pred_sm))
print("Precision:", precision_score(y_test, y_pred_sm))
print("Recall:", recall_score(y_test, y_pred_sm))
print("F1:", f1_score(y_test, y_pred_sm))

After SMOTE:
Accuracy: 0.66
Precision: 0.30303030303030304
Recall: 0.47619047619047616
F1: 0.37037037037037035


In [17]:
print("--- Without SMOTE ---")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

print("--- With SMOTE ---")
print(confusion_matrix(y_test, y_pred_sm))
print(classification_report(y_test, y_pred_sm))

--- Without SMOTE ---
[[71  8]
 [18  3]]
              precision    recall  f1-score   support

           0       0.80      0.90      0.85        79
           1       0.27      0.14      0.19        21

    accuracy                           0.74       100
   macro avg       0.54      0.52      0.52       100
weighted avg       0.69      0.74      0.71       100

--- With SMOTE ---
[[56 23]
 [11 10]]
              precision    recall  f1-score   support

           0       0.84      0.71      0.77        79
           1       0.30      0.48      0.37        21

    accuracy                           0.66       100
   macro avg       0.57      0.59      0.57       100
weighted avg       0.72      0.66      0.68       100

